# Amazon ML Challenge 2026 — EDA & Preprocessing (final)

Business Entity Resolution: match noisy records in **S2 / S3** back to the reference entities in **S1**.
This notebook writes the cleaned files that `blocking_final.py`, `feature_engineering.py` and `final.py` read:
`artifacts/{train,test}_s{1,2,3}_clean.csv`, `artifacts/ground_truth_clean.csv`, `artifacts/train_val_split.csv`.

It keeps your friend's approach (romanise first, skeleton-aware legal forms, state normalisation,
alias split, address components; same 50 columns) and fixes what made the same entity produce
**different blocking keys** on the two sides:

| # | problem seen in the blocking misses | fix |
|---|---|---|
| 1 | `Holloway`, `Ellis`, `Wallace`, `Cyril` deleted as if they were `llp`, `llc`, `pllc`, `sarl` | phonetic legal-form stripping only for names written in a non-Latin script, or for words a typo away from a legal word (`limted`, `privte`) |
| 2 | `OUTSHlNE`, `di6ital`, `c0mpany` | look-alike letters / digits repaired inside words |
| 3 | `candy's`, `jacky’s` kept the apostrophe (`jacky?s` never matched `jackysinsurancecom`) | apostrophes removed, French elision split (`l'atelier` → `l atelier`) |
| 4 | `abc.co.in`, `www.abc.in` not recognised as web names | every web name becomes `abccom` (blocking splits it into words) |
| 5 | `pra li` (= pvt ltd) left in transliterated names | trailing transliterated legal abbreviations removed |
| 6 | `225vivek`, `1161d`, `no94`, `35th`, `00701`, `##97` | split, ordinals / leading zeros removed, masked digits kept as `97` **and** `xx97` |
| 7 | `19034 Woodburn Rd` → postal code 19034 | postal code = last 5-6 digit number that is not the first token |
| 8 | `null` / `nan` / `cedex` address words, mojibake (`CafÃ©`) | dropped / repaired |
| 9 | French legal forms and regions unknown (test has France) | added to the vocabulary |

**No rule looks at the `country` column** — every record gets the same cleaning.

The cleaning code is written to `prep_final.py` (next cells) so `final.py` can use exactly the same
functions for the test files. Run the whole notebook top to bottom.

In [1]:
import gc
import importlib
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

# the folder that holds train/ and test/ (this notebook normally sits in it)
DATA_DIR = next((p for p in (Path("."), Path("dataset"), Path("..")) if (p / "train" / "train_source1.tsv").exists()),
                Path("."))
TRAIN_DIR, TEST_DIR = DATA_DIR / "train", DATA_DIR / "test"
ART = DATA_DIR / "artifacts"
ART.mkdir(exist_ok=True)
PREP_PATH = DATA_DIR / "prep_final.py"
print("data folder :", DATA_DIR.resolve())
print("artifacts   :", ART.resolve())

data folder : /Users/mohnish/Desktop/AMAZON/student_resource/dataset
artifacts   : /Users/mohnish/Desktop/AMAZON/student_resource/dataset/artifacts


## 0. The cleaning code

The next cell writes `prep_final.py` next to `blocking_final.py`. Edit the cleaning **here** and re-run
the notebook; do not edit the .py file by hand.

In [2]:
%%writefile {PREP_PATH}
# ============================================================================
#  prep_final.py — preprocessing for the Amazon ML Challenge 2026
#                  (Business Entity Resolution)
#
#  Written by 0_eda_preprocessing_final.ipynb. Edit the notebook, not this file:
#  re-running the notebook rewrites it. final.py can import it so the test files
#  get exactly the same cleaning as train.
#
#  Base: the friend's utils (romanise first, skeleton-aware legal forms, state
#  normalisation, alias split, address components). Changes, all aimed at
#  making the same entity produce the same blocking keys on both sides:
#
#  NAMES
#   1. mojibake repaired ("CafÃ©" -> "Café") before anything else
#   2. look-alike characters: "OUTSHlNE" -> outshine, "di6ital" -> digital,
#      "c0mpany" -> company (letters inside words only; codes such as "b2b" kept)
#   3. web names in one canonical form: "abc.co.in", "www.abc.in", "abc.com"
#      -> "abccom"  (blocking splits "...com" words back into S1 words)
#   4. apostrophes: "candy's" -> candys, "jacky’s" -> jackys, "l'atelier" -> l atelier
#   5. legal forms stripped by their phonetic skeleton only for names written in a
#      non-Latin script, or (Latin script) when the word is also within a few edits
#      of the legal word ("limted", "privte"). Before, "Holloway", "Ellis",
#      "Wallace", "Cyril" were deleted as if they were "llp", "llc", "pllc", "sarl".
#      Trailing transliterated abbreviations ("pra li" = pvt ltd) are stripped too.
#   6. more legal forms (sasu, eurl, selarl, gmbh, plc, ...), "et" / "&" -> and
#   7. alias markers "t/a", "d/b/a", "a.k.a." recognised; the marker word itself
#      is removed from name_core
#  ADDRESSES
#   8. masked / labelled digits: "##97" -> 97 + xx97, "#4" -> 4, "N°12" -> no 12
#   9. glued tokens split: "225vivek" -> 225 vivek, "1161d" -> 1161 d, "no94" -> no 94
#  10. ordinals and leading zeros: "35th" -> 35, "00701" -> 701
#  11. junk words dropped: null, nan, none, cedex, bis
#  12. postal code: the last 5-6 digit number that is NOT the first token
#      (a leading "19034 Woodburn Rd" is a house number, not a ZIP)
#  13. more street / building words and region names (French added)
#
#  Nothing here looks at the country column: every record gets the same rules.
# ============================================================================
import os
import re
import unicodedata
from concurrent.futures import ProcessPoolExecutor
from functools import lru_cache

import numpy as np
import pandas as pd
from unidecode import unidecode
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

# parallel cleaning of unique strings (1 = off).  Each worker cleans a slice.
WORKERS = max(1, min(8, (os.cpu_count() or 2) - 1))
PAR_MIN = 200_000            # fewer unique strings than this -> single process


# ============================ COLUMN 1 AND 4 ============================
def add_source_column(df: pd.DataFrame) -> pd.DataFrame:
    """Derive source (S1/S2/S3) from the entity_id prefix."""
    df = df.copy()
    df["source"] = df["entity_id"].astype(str).str.split("-").str[0]
    return df


def normalize_country(df: pd.DataFrame) -> pd.DataFrame:
    """Whitespace cleanup only (country is a hard blocking filter)."""
    df = df.copy()
    df["country_clean"] = (df["country"].astype(str).str.strip()
                           .str.replace(r"\s+", " ", regex=True))
    return df


# ============================ shared text helpers ============================
_MOJI = re.compile("[\u0080-\u009f]|[Â-ô][\u0080-¿]")


def _fix_mojibake(s: str) -> str:
    """UTF-8 text that was decoded as cp1252 / latin-1 ('CafÃ©', 'itâ€™s')."""
    if s.isascii() or not _MOJI.search(s):
        return s
    for enc in ("cp1252", "latin-1"):
        try:
            return s.encode(enc).decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
    return s


_INDIC_BLOCKS = [
    (0x0900, 0x097F, sanscript.DEVANAGARI),
    (0x0980, 0x09FF, sanscript.BENGALI),
    (0x0A00, 0x0A7F, sanscript.GURMUKHI),
    (0x0A80, 0x0AFF, sanscript.GUJARATI),
    (0x0B00, 0x0B7F, sanscript.ORIYA),
    (0x0B80, 0x0BFF, sanscript.TAMIL),
    (0x0C00, 0x0C7F, sanscript.TELUGU),
    (0x0C80, 0x0CFF, sanscript.KANNADA),
    (0x0D00, 0x0D7F, sanscript.MALAYALAM),
]


def _dominant_indic_scheme(text):
    counts = {}
    for ch in text:
        cp = ord(ch)
        if cp < 0x0900 or cp > 0x0D7F:
            continue
        for lo, hi, scheme in _INDIC_BLOCKS:
            if lo <= cp <= hi:
                counts[scheme] = counts.get(scheme, 0) + 1
                break
    return max(counts, key=counts.get) if counts else None


def _non_latin(text: str) -> bool:
    """True if the text has letters from a non-Latin script (Indic, Arabic, ...)."""
    for ch in text:
        if ord(ch) >= 0x0370 and ch.isalpha():
            return True
    return False


def _romanize(text):
    """Latin ASCII: transliterate the INTACT original-script string first (so vowel
    signs survive), then unidecode flattens diacritics / accents."""
    if text.isascii():
        return text
    scheme = _dominant_indic_scheme(text)
    if scheme is not None:
        try:
            text = transliterate(text, scheme, sanscript.IAST)
        except Exception:
            pass
    return unidecode(text)


_QUOTES = re.compile(r"[`´‘’ʼ′]")
_ELISION = re.compile(r"\b(qu|l|d)'(?=[a-z])", re.I)


def _apostrophes(text: str) -> str:
    """'l'atelier' -> 'l atelier';  every other apostrophe is removed ('candy's' -> candys)."""
    text = _QUOTES.sub("'", text)
    if "'" not in text:
        return text
    text = _ELISION.sub(r"\1 ", text)
    return text.replace("'", "")


# ============================ COLUMN 2 : business_name ============================
# every spelling -> canonical legal-form word (vocabulary, not a per-country rule)
SUFFIX_MAP = {
    "incorporated": "inc", "inc": "inc",
    "corporation": "corp", "corp": "corp",
    "company": "co", "co": "co",
    "limited": "ltd", "ltd": "ltd",
    "private": "pvt", "pvt": "pvt",
    "llc": "llc", "llp": "llp", "pllc": "pllc", "lp": "lp",
    "sarl": "sarl", "sas": "sas", "sasu": "sasu", "eurl": "eurl", "sa": "sa",
    "selarl": "selarl", "snc": "snc",
    "gmbh": "gmbh", "plc": "plc", "pty": "pty", "pte": "pte", "bhd": "bhd",
    "sdn": "sdn", "srl": "srl", "opc": "opc",
}
SUFFIX_CANONICAL = set(SUFFIX_MAP.values())

_SKEL_DROP = set("aeiouhy")
_SKEL_MAP = {
    **{c: "b" for c in "bvwpf"},
    **{c: "s" for c in "cszjx"},
    **{c: "d" for c in "dt"},
    **{c: "k" for c in "kgq"},
    **{c: c for c in "mnrl"},
}


@lru_cache(maxsize=1_000_000)
def phonetic_skeleton(token: str) -> str:
    """script-agnostic consonant skeleton:  private / praiveta -> brbd."""
    return "".join(_SKEL_MAP.get(ch, ch if ch.isdigit() else "")
                   for ch in token if ch not in _SKEL_DROP)


_SUFFIX_WORDS = ["private", "limited", "pvt", "ltd", "company", "corporation",
                 "corp", "incorporated", "llc", "llp", "pllc", "sarl", "kampani"]
SUFFIX_SKELETONS = {s for s in (phonetic_skeleton(w) for w in _SUFFIX_WORDS) if len(s) >= 3}
_SKEL_WORDS = {}
for _w in _SUFFIX_WORDS:
    _SKEL_WORDS.setdefault(phonetic_skeleton(_w), []).append(_w)


def _lev(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


@lru_cache(maxsize=200_000)
def _typo_suffix(tok: str) -> bool:
    """a Latin-script word counts as a (misspelt) legal form only if it is within a
    few edits of one: 'limted', 'privte', 'corporaton' yes;  'holloway' (skeleton of
    llp), 'ellis' (llc), 'wallace' (pllc), 'cyril' (sarl) no."""
    for w in _SKEL_WORDS.get(phonetic_skeleton(tok), ()):
        if len(w) >= 5 and _lev(tok, w) <= max(1, len(w) // 4):
            return True
    return False
# trailing transliterated abbreviations of a non-Latin-script name: "pra li" = pvt ltd
_SHORT_SUFFIX_SKELETONS = {"br", "l", "lmd", "brb"}

_TOKRE = re.compile(r"[A-Za-z0-9]+")


def _fix_case_lookalikes(text: str) -> str:
    """'OUTSHlNE' -> 'OUTSHINE' (lower-case l inside an upper-case word) and
    'heIIo' -> 'hello' (upper-case I inside a lower-case word)."""
    def fix(m):
        t = m.group(0)
        if len(t) < 3:
            return t
        if "l" in t:
            rest = t.replace("l", "")
            if sum(c.isupper() for c in rest) >= 2 and not any(c.islower() for c in rest):
                return t.replace("l", "I")
        if "I" in t and t[0].islower():
            rest = t.replace("I", "")
            if sum(c.islower() for c in rest) >= 2 and not any(c.isupper() for c in rest):
                return t.replace("I", "l")
        return t
    return _TOKRE.sub(fix, text)


_WEB_RE = re.compile(r"(?:https?://)?(?:www\.)?([a-z0-9][a-z0-9\-]*)\.(?:co\.in|co\.uk|com|net|org|in|co|"
                     r"biz|info|us|fr|io)\b", re.I)


def _web_names(text: str) -> str:
    """every web name in one form: 'abc.co.in' / 'www.abc.in' / 'abc.com' -> 'abccom'."""
    if "." not in text:
        return text
    return _WEB_RE.sub(lambda m: m.group(1).replace("-", "") + "com", text)


_DIG2LET = str.maketrans("013456789", "oleasgtbg")
_ORD_TOK = re.compile(r"^\d+(st|nd|rd|th)$")


def _fix_digit_lookalikes(tok: str) -> str:
    """'di6ital' -> digital, 'sik0ra' -> sikora, '8akery' -> bakery, 'federa1' -> federal.
    Only words with >= 3 letters and at most 2 isolated digits (codes like 'abc12',
    'b2b', '24x7' and ordinals are left alone)."""
    if tok.isalpha() or tok.isdigit():
        return tok
    nd = sum(c.isdigit() for c in tok)
    if nd > 2 or len(tok) - nd < 3 or _ORD_TOK.match(tok):
        return tok
    for i in range(len(tok) - 1):
        if tok[i].isdigit() and tok[i + 1].isdigit():
            return tok
    return tok.translate(_DIG2LET)


_ALIAS_RE = re.compile(r"\b(?:d\s*/\s*b\s*/\s*a|t\s*/\s*a|d\.\s*b\.\s*a\.?|a\.\s*k\.\s*a\.?|f\.\s*k\.\s*a\.?|"
                       r"doing business as)(?=\s|$|[^a-z])", re.I)
_ALIAS_1TOK = {"dba", "aka", "fka"}
_ALIAS_2TOK = (("trading", "as"), ("also", "known"), ("formerly", "known"))
_PUNCT_NAME = re.compile(r"[^a-z0-9 ]")


def _tokenize_name(text: str):
    text = text.replace("&", " and ").replace("+", " and ").replace(".", "")
    text = _PUNCT_NAME.sub(" ", text.lower())
    out = []
    for t in text.split():
        t = _fix_digit_lookalikes(t)
        if t == "et":
            t = "and"
        t = SUFFIX_MAP.get(t, t)
        if not out or out[-1] != t:                       # collapse consecutive duplicates
            out.append(t)
    return out


def _is_suffix_token(tok: str, script: bool) -> bool:
    if tok in SUFFIX_CANONICAL:
        return True
    if phonetic_skeleton(tok) not in SUFFIX_SKELETONS:
        return False
    return script or _typo_suffix(tok)


def _core_of(toks, script=False):
    """drop legal-form words (never empty the name)."""
    core = [t for t in toks if not _is_suffix_token(t, script)]
    if script and len(core) > 2:                          # trailing transliterated "pra li" / "pri lim"
        sk = [phonetic_skeleton(t) for t in core[-2:]]
        if sk[0] == "br" and sk[1] in ("l", "lmd") and len(core[-1]) <= 4:
            core = core[:-2]
    if not core:
        core = [t for t in toks if t not in SUFFIX_CANONICAL]
    if not core:
        core = list(toks)
    return core


def _split_alias(toks):
    """(primary_toks, alias_toks, marker_positions).  Alias words are empty when no
    marker has words on both sides; marker words are always reported so they can
    leave the name."""
    n = len(toks)
    marks = set()
    split = None
    i = 0
    while i < n:
        t = toks[i]
        if t in _ALIAS_1TOK:
            marks.add(i)
            if split is None and 0 < i < n - 1:
                split = (i, i + 1)
        elif i + 1 < n and (t, toks[i + 1]) in _ALIAS_2TOK:
            j = i + 2
            if j < n and toks[j] == "as":
                j += 1
            if split is None and 0 < i and j < n:
                marks.update(range(i, j))
                split = (i, j)
            i = j
            continue
        elif t == "formerly" and split is None and 0 < i < n - 1:
            marks.add(i)
            split = (i, i + 1)
        i += 1
    if split is None:
        return toks, [], marks
    a, b = split
    prim = [t for k, t in enumerate(toks[:a]) if k not in marks]
    alias = [t for k, t in enumerate(toks[b:], start=b) if k not in marks]
    return prim, alias, marks


def _name_shape(core_toks):
    compact = "".join(core_toks)
    num_toks = [t for t in core_toks if t.isdigit()]
    alnum_toks = [t for t in core_toks if any(c.isdigit() for c in t) and not t.isdigit()]
    return (len(core_toks), compact, len(compact),
            core_toks[0] if core_toks else "", core_toks[-1] if core_toks else "",
            "".join(t[0] for t in core_toks if t), bool(num_toks or alnum_toks),
            ",".join(num_toks), ",".join(alnum_toks))


def clean_business_name(raw):
    text = _fix_mojibake(str(raw))
    text = unicodedata.normalize("NFKC", text)
    text = text.split("|")[0]                              # "NAME | www.site.com"
    text = re.sub(r"^[\s\-]+", "", text)                   # leading "-- " junk
    script = _non_latin(text)
    if text.isascii():
        text = _fix_case_lookalikes(text)
    text = _web_names(text)
    text = _ALIAS_RE.sub(" dba ", text)
    text = _apostrophes(_romanize(text))
    toks = _tokenize_name(text)

    primary, alias, marks = _split_alias(toks)
    body = [t for i, t in enumerate(toks) if i not in marks]   # the marker word leaves the name
    core = _core_of(body, script)
    has_alias = bool(alias)
    name_primary = " ".join(_core_of(primary, script)) if has_alias else ""
    name_alias = " ".join(_core_of(alias, script)) if has_alias else ""

    (n_tok, compact, char_len, first_tok, last_tok,
     initials, has_digit, num_toks, alnum_toks) = _name_shape(core)
    name_skeleton = "".join(sorted(s for s in (phonetic_skeleton(t) for t in core) if s))
    return (
        " ".join(body),            # name_translit   (romanised, legal forms canonical)
        " ".join(core),            # name_core       (legal forms removed)  <- blocking
        " ".join(sorted(core)),    # name_tokens_sorted
        len(core) < len(body),     # has_legal_suffix
        n_tok, compact, char_len, first_tok, last_tok, initials, has_digit,
        num_toks, alnum_toks, name_skeleton, has_alias, name_primary, name_alias,
    )


_NAME_COLS = [
    "name_translit", "name_core", "name_tokens_sorted", "has_legal_suffix",
    "name_token_count", "name_compact", "name_char_len", "name_first_token",
    "name_last_token", "name_initials", "name_has_digit", "name_numeric_tokens",
    "name_alnum_tokens", "name_skeleton", "has_alias", "name_primary", "name_alias",
]


# ============================ COLUMN 3 : business_address ============================
ADDR_ABBR = {
    # street / place types
    "road": "rd", "rd": "rd",
    "street": "st", "st": "st", "str": "st",
    "avenue": "ave", "ave": "ave", "av": "ave",
    "boulevard": "blvd", "blvd": "blvd", "boul": "blvd", "bd": "blvd", "bld": "blvd",
    "lane": "ln", "ln": "ln",
    "drive": "dr", "dr": "dr",
    "court": "ct", "ct": "ct",
    "circle": "cir", "cir": "cir",
    "place": "pl", "pl": "pl",
    "square": "sq", "sq": "sq",
    "highway": "hwy", "hwy": "hwy",
    "parkway": "pkwy", "pkwy": "pkwy",
    "terrace": "ter", "ter": "ter",
    "trail": "trl", "trl": "trl",
    "expressway": "expy", "expy": "expy", "freeway": "fwy", "fwy": "fwy",
    "suite": "ste", "ste": "ste",
    "apartment": "apt", "apartments": "apt", "apt": "apt", "appartement": "apt", "appt": "apt",
    "building": "bldg", "bldg": "bldg", "bhawan": "bldg", "bhavan": "bldg",
    "batiment": "bldg", "bat": "bldg",
    "floor": "fl", "flr": "fl", "fl": "fl", "etage": "fl",
    "number": "no", "no": "no", "num": "no", "nos": "no",
    "block": "blk", "blk": "blk",
    "opposite": "opp", "opp": "opp",
    "center": "ctr", "centre": "ctr", "ctr": "ctr",
    "mount": "mt", "mt": "mt", "fort": "ft", "ft": "ft", "point": "pt", "pt": "pt",
    "heights": "hts", "hts": "hts",
    "saint": "st", "sainte": "st",
    "north": "n", "south": "s", "east": "e", "west": "w",
    "northeast": "ne", "northwest": "nw", "southeast": "se", "southwest": "sw",
    # sub-units / land records
    "house": "hno", "hno": "hno", "hn": "hno",
    "plot": "plot", "plt": "plot",
    "sector": "sector", "sec": "sector",
    "phase": "phase",
    "post": "po", "po": "po",
    "district": "dist", "dist": "dist",
    "khasra": "kh", "kh": "kh",
    # other languages
    "rue": "rue",
    "impasse": "imp", "imp": "imp",
    "chemin": "chem", "chem": "chem",
    "route": "rte", "rte": "rte",
    "quai": "quai",
    "residence": "res", "res": "res",
    "faubourg": "fbg", "fbg": "fbg",
}

_US_STATES = {
    "alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar",
    "california": "ca", "colorado": "co", "connecticut": "ct", "delaware": "de",
    "florida": "fl", "georgia": "ga", "hawaii": "hi", "idaho": "id",
    "illinois": "il", "indiana": "in", "iowa": "ia", "kansas": "ks",
    "kentucky": "ky", "louisiana": "la", "maine": "me", "maryland": "md",
    "massachusetts": "ma", "michigan": "mi", "minnesota": "mn",
    "mississippi": "ms", "missouri": "mo", "montana": "mt", "nebraska": "ne",
    "nevada": "nv", "new hampshire": "nh", "new jersey": "nj",
    "new mexico": "nm", "new york": "ny", "north carolina": "nc",
    "north dakota": "nd", "ohio": "oh", "oklahoma": "ok", "oregon": "or",
    "pennsylvania": "pa", "rhode island": "ri", "south carolina": "sc",
    "south dakota": "sd", "tennessee": "tn", "texas": "tx", "utah": "ut",
    "vermont": "vt", "virginia": "va", "washington": "wa",
    "west virginia": "wv", "wisconsin": "wi", "wyoming": "wy",
    "district of columbia": "dc",
}
_IN_STATES = {
    "andhra pradesh": "ap", "andhra pradesa": "ap",
    "arunachal pradesh": "ar",
    "assam": "as", "asama": "as",
    "bihar": "br", "bihara": "br",
    "chhattisgarh": "cg", "chhattisagadha": "cg", "chattisgarh": "cg",
    "goa": "ga",
    "gujarat": "gj", "gujarata": "gj",
    "haryana": "hr", "hariyana": "hr",
    "himachal pradesh": "hp", "himachala pradesa": "hp",
    "jharkhand": "jh", "jharakhanda": "jh",
    "karnataka": "ka", "karnatak": "ka",
    "kerala": "kl", "keralam": "kl",
    "madhya pradesh": "mp", "madhya pradesa": "mp",
    "maharashtra": "mh", "maharastra": "mh",
    "manipur": "mn", "manipura": "mn",
    "meghalaya": "ml", "mizoram": "mz", "nagaland": "nl",
    "odisha": "od", "orissa": "od", "odisa": "od",
    "punjab": "pb", "pamjaba": "pb", "panjaba": "pb",
    "rajasthan": "rj", "rajasthana": "rj",
    "sikkim": "sk",
    "tamil nadu": "tn", "tamilnadu": "tn", "dhamilnadhu": "tn", "tamila nadu": "tn",
    "telangana": "tg", "telamgana": "tg",
    "tripura": "tr",
    "uttar pradesh": "up", "uttara pradesa": "up",
    "uttarakhand": "uk", "uttarakhanda": "uk", "uttaranchal": "uk",
    "west bengal": "wb", "pascimavanga": "wb", "pascimabangg": "wb", "pascima bangala": "wb",
    "delhi": "dl", "new delhi": "dl", "dilli": "dl",
    "jammu and kashmir": "jk", "ladakh": "la",
    "puducherry": "py", "pondicherry": "py", "chandigarh": "ch",
}
_FR_REGIONS = {
    "ile de france": "idf", "auvergne rhone alpes": "ara", "nouvelle aquitaine": "naq",
    "occitanie": "occ", "hauts de france": "hdf", "provence alpes cote d azur": "pac",
    "provence alpes cote dazur": "pac", "paca": "pac", "grand est": "ges",
    "pays de la loire": "pdl", "normandie": "nor", "bretagne": "bre",
    "bourgogne franche comte": "bfc", "centre val de loire": "cvl", "corse": "cor",
}
_STATE_MAP = {**_US_STATES, **_IN_STATES, **_FR_REGIONS}
_STATE_RE = re.compile(r"\b(" + "|".join(sorted((re.escape(k) for k in _STATE_MAP),
                                                key=len, reverse=True)) + r")\b")


def _norm_states(text: str):
    """'north carolina' -> 'nc', 'maharastra' -> 'mh';  also returns whether one was found."""
    found = _STATE_RE.search(text) is not None
    return (_STATE_RE.sub(lambda m: _STATE_MAP[m.group(0)], text) if found else text), found


_NUM56 = re.compile(r"(?<!\d)(\d{5,6})(?!\d)")
_PIN_SPLIT = re.compile(r"(?<!\d)(\d{3})\s(\d{3})\s*$")
_ALPHA = re.compile(r"[^\W\d_]")


def _extract_pin(text):
    """postal code = the LAST 5-6 digit number that has a word before it
    (a number that starts the address is its house number)."""
    best = ""
    for m in _NUM56.finditer(text):
        if _ALPHA.search(text, 0, m.start()):
            best = m.group(1)
    if not best:
        m = _PIN_SPLIT.search(text)                        # "110 063"
        if m and _ALPHA.search(text, 0, m.start()):
            best = m.group(1) + m.group(2)
    return best


_NUMERO = re.compile(r"\b[nN]\s*[°º]\s*")
_MASKED = re.compile(r"(#{2,})\s*(\d+)")
_HNO = re.compile(r"\bh\s*\.?\s*no\b\.?")
_PO = re.compile(r"\bp\s*\.\s*o\b\.?")
_DOTTED = re.compile(r"\b[a-z](?:\.[a-z])+\b\.?")          # m.g. road -> mg road
_ORD = re.compile(r"\b(\d+)(?:st|nd|rd|th)\b")
_GLUE = re.compile(r"(?<=\d)(?=[a-z])|(?<=[a-z])(?=\d)")
_LEAD0 = re.compile(r"\b0+(?=\d)")
_PUNCT_ADDR = re.compile(r"[^a-z0-9 ]")
_JUNK_ADDR = {"null", "nan", "none", "cedex", "bis"}

# ---------------------------------------------------------------- components
_NUMLIKE = re.compile(r"^\d+[a-z]?$|^[a-z]-?\d+[a-z]?$", re.I)
_BLOCK_VAL = re.compile(r"^[a-z0-9]{1,3}$", re.I)


def _value_ok(field, tok):
    return _BLOCK_VAL.match(tok) if field == "block" else _NUMLIKE.match(tok)


_LABEL_FIELD = {
    "hno": "house_number",
    "plot": "plot_number", "kh": "plot_number",
    "sector": "sector", "phase": "phase", "blk": "block",
    "apt": "unit_number", "ste": "unit_number",
}
_FILLER_AFTER_LABEL = {"no"}
_NO_CONTEXT = {
    "shop": "unit_number", "flat": "unit_number", "door": "unit_number",
    "office": "unit_number", "unit": "unit_number", "room": "unit_number",
    "house": "house_number", "building": "house_number", "bldg": "house_number",
}
_FLOOR_WORD_RE = re.compile(r"^(\d+)(st|nd|rd|th)?$")
STREET_TYPES = {"rd", "st", "ave", "blvd", "ln", "dr", "ct", "cir", "pl", "sq",
                "hwy", "pkwy", "ter", "trl", "expy", "fwy", "rue", "imp", "chem", "rte", "quai"}
LANDMARK_MARKERS = {"near", "opp", "behind", "beside", "adjacent", "next",
                    "infront", "front", "above", "below", "close", "backside"}


def _extract_address_components(toks):
    """best-effort labelled components (empty when unsure)."""
    n = len(toks)
    comp = {"house_number": "", "unit_number": "", "floor_number": "",
            "plot_number": "", "sector": "", "phase": "", "block": "",
            "street_name": "", "landmark": ""}
    if n == 0:
        comp["address_core"] = ""
        return comp
    used = [False] * n
    for i, t in enumerate(toks):
        field = _LABEL_FIELD.get(t)
        if field and not comp[field]:
            j = i + 1
            if j < n and toks[j] in _FILLER_AFTER_LABEL:
                j += 1
            if j < n and not used[j] and _value_ok(field, toks[j]):
                comp[field] = toks[j]
                used[i] = used[j] = True
                if j != i + 1:
                    used[i + 1] = True
        elif t == "fl" and not comp["floor_number"]:
            for j in (i - 1, i + 1):
                if 0 <= j < n and not used[j] and _FLOOR_WORD_RE.match(toks[j]):
                    comp["floor_number"] = _FLOOR_WORD_RE.match(toks[j]).group(1)
                    used[i] = used[j] = True
                    break
        elif t == "no" and i > 0 and not used[i - 1] and not used[i] and toks[i - 1] in _NO_CONTEXT:
            field = _NO_CONTEXT[toks[i - 1]]
            if not comp[field] and i + 1 < n and not used[i + 1] and _NUMLIKE.match(toks[i + 1]):
                comp[field] = toks[i + 1]
                used[i - 1] = used[i] = used[i + 1] = True

    landmark_idx = set()
    for i, t in enumerate(toks):
        if t in LANDMARK_MARKERS and not used[i]:
            span = [t]
            landmark_idx.add(i)
            for j in range(i + 1, min(i + 5, n)):
                if used[j] or toks[j] in LANDMARK_MARKERS or toks[j] in _LABEL_FIELD:
                    break
                span.append(toks[j])
                landmark_idx.add(j)
            comp["landmark"] = " ".join(span)
            break
    for i in landmark_idx:
        used[i] = True

    if not comp["house_number"] and not used[0]:
        if _NUMLIKE.match(toks[0]):
            comp["house_number"] = toks[0]
            used[0] = True
        elif len(toks[0]) == 1 and toks[0].isalpha() and n > 1 and not used[1] and toks[1].isdigit():
            comp["house_number"] = toks[0] + toks[1]
            used[0] = used[1] = True

    for i, t in enumerate(toks):
        if t in STREET_TYPES and not used[i]:
            j = i - 1
            words = []
            while (j >= 0 and len(words) < 2 and not used[j] and not _NUMLIKE.match(toks[j])
                   and len(toks[j]) > 1 and toks[j] not in _LABEL_FIELD and toks[j] not in STREET_TYPES):
                words.append(toks[j])
                j -= 1
            if words:
                comp["street_name"] = " ".join(reversed(words)) + " " + t
                used[i] = True
            break

    comp["address_core"] = " ".join(t for i, t in enumerate(toks) if i not in landmark_idx)
    return comp


_EMPTY_ADDR = ("", "", "", False, "", "", "", "", "", "", "", "", "", "", 0, 0, "", 0)


def _clean_address(raw):
    if not isinstance(raw, str) or raw.strip() == "":
        return _EMPTY_ADDR
    text = unicodedata.normalize("NFKC", _fix_mojibake(raw))
    text = _NUMERO.sub(" no ", text).replace("°", " ").replace("º", " ")
    # masked digits "##97": keep the visible number AND its masked form "xx97" (blocking
    # gives every S1 number the same masked form: 1297 -> xx97);  "#4" -> 4
    masks = ["x" * len(m.group(1)) + m.group(2) for m in _MASKED.finditer(text)]
    text = text.replace("#", " ")
    pin = _extract_pin(text)
    roman = _apostrophes(_romanize(text)).lower()
    roman = _HNO.sub(" hno ", roman)
    roman = _PO.sub(" po ", roman)
    roman = _DOTTED.sub(lambda m: m.group(0).replace(".", ""), roman)
    roman = _PUNCT_ADDR.sub(" ", roman.replace("&", " and "))
    roman = _ORD.sub(r"\1", roman)                         # 35th -> 35
    roman = _GLUE.sub(" ", roman)                          # 225vivek -> 225 vivek, 1161d -> 1161 d
    roman = _LEAD0.sub("", roman)                          # 00701 -> 701
    roman = " ".join(t for t in roman.split() if t not in _JUNK_ADDR)
    roman, has_state = _norm_states(roman)
    out = []
    for t in roman.split():
        t = ADDR_ABBR.get(t, t)
        if not out or out[-1] != t:
            out.append(t)
    toks = out + [m for m in masks if m not in out]
    address_clean = " ".join(toks)
    comp = _extract_address_components(toks)
    num_toks = sorted({t for t in toks if t.isdigit()})
    return (
        address_clean, pin, " ".join(sorted(toks)), has_state,
        comp["house_number"], comp["unit_number"], comp["floor_number"], comp["plot_number"],
        comp["sector"], comp["phase"], comp["block"], comp["street_name"], comp["landmark"],
        comp["address_core"], len(toks), len(address_clean), ",".join(num_toks), len(num_toks),
    )


_ADDR_COLS = [
    "address_clean", "address_pin", "address_tokens_sorted", "has_state_token",
    "house_number", "unit_number", "floor_number", "plot_number", "sector",
    "phase", "block", "street_name", "landmark", "address_core",
    "address_token_count", "address_char_len", "address_numeric_tokens",
    "address_numeric_count",
]


# ============================ table-level processing ============================
_FUNCS = {"name": clean_business_name, "addr": _clean_address}


def _clean_chunk(args):
    kind, values = args
    fn = _FUNCS[kind]
    rows = [fn(v) for v in values]
    return [list(col) for col in zip(*rows)] if rows else None


def _clean_unique(values: pd.Series, kind: str, ncols: int, workers=None):
    """clean every UNIQUE string once (in parallel when there are many) ->
    list of ncols object arrays aligned with `values`."""
    codes, uniq = pd.factorize(values, sort=False)
    uniq = list(uniq)
    workers = WORKERS if workers is None else workers
    if workers > 1 and len(uniq) >= PAR_MIN:
        parts = [uniq[i::workers * 4] for i in range(workers * 4)]
        with ProcessPoolExecutor(max_workers=workers) as ex:
            res = list(ex.map(_clean_chunk, [(kind, p) for p in parts], chunksize=1))
        cols = [np.empty(len(uniq), dtype=object) for _ in range(ncols)]
        k = workers * 4
        for i, r in enumerate(res):
            if r is None:
                continue
            for c in range(ncols):
                cols[c][i::k] = r[c]
    else:
        r = _clean_chunk((kind, uniq))
        cols = [np.array(c, dtype=object) for c in r] if r else [np.empty(0, object)] * ncols
    codes = np.where(codes < 0, 0, codes)
    return [c[codes] if len(c) else np.empty(len(values), object) for c in cols]


def process_business_name(df: pd.DataFrame, workers=None) -> pd.DataFrame:
    df = df.drop(columns=[c for c in _NAME_COLS if c in df.columns])
    names = df["business_name"].fillna("").astype(str)
    cols = _clean_unique(names, "name", len(_NAME_COLS), workers)
    for name, col in zip(_NAME_COLS, cols):
        df[name] = col
    for c in ("has_legal_suffix", "name_has_digit", "has_alias"):
        df[c] = df[c].astype(bool)
    for c in ("name_token_count", "name_char_len"):
        df[c] = df[c].astype(np.int32)
    return df


def process_business_address(df: pd.DataFrame, workers=None) -> pd.DataFrame:
    extra = _ADDR_COLS + ["is_missing_address"] + [
        f"has_{c}" for c in ("house_number", "unit_number", "floor_number",
                             "plot_number", "sector", "street_name", "landmark")] + ["has_postal_code"]
    df = df.drop(columns=[c for c in extra if c in df.columns])
    s = df["business_address"].fillna("").astype(str)
    cols = _clean_unique(s, "addr", len(_ADDR_COLS), workers)
    for name, col in zip(_ADDR_COLS, cols):
        df[name] = col
    df["has_state_token"] = df["has_state_token"].astype(bool)
    for c in ("address_token_count", "address_char_len", "address_numeric_count"):
        df[c] = df[c].astype(np.int32)
    df["is_missing_address"] = (df["address_clean"] == "").to_numpy()
    for c in ("house_number", "unit_number", "floor_number", "plot_number",
              "sector", "street_name", "landmark"):
        df[f"has_{c}"] = df[c] != ""
    df["has_postal_code"] = df["address_pin"] != ""
    return df


# ============================ diagnostics ============================
SCRIPT_RANGES = {
    "Devanagari": (0x0900, 0x097F), "Bengali": (0x0980, 0x09FF),
    "Gurmukhi": (0x0A00, 0x0A7F), "Gujarati": (0x0A80, 0x0AFF),
    "Oriya": (0x0B00, 0x0B7F), "Tamil": (0x0B80, 0x0BFF),
    "Telugu": (0x0C00, 0x0C7F), "Kannada": (0x0C80, 0x0CFF),
    "Malayalam": (0x0D00, 0x0D7F), "Latin-Extended-Accented": (0x00C0, 0x024F),
    "Arabic": (0x0600, 0x06FF), "Cyrillic": (0x0400, 0x04FF), "CJK": (0x4E00, 0x9FFF),
}


def scan_column_scripts(datasets: dict, col: str) -> pd.DataFrame:
    rows = []
    for table_name, df in datasets.items():
        if col not in df.columns:
            continue
        s = df[col].dropna().astype(str)
        non_ascii = ~s.str.match(r"^[\x00-\x7F]*$")
        row = {"table": table_name, "total": len(s), "non_ascii": int(non_ascii.sum())}
        accounted = pd.Series(False, index=s.index)
        for script, (lo, hi) in SCRIPT_RANGES.items():
            mask = s.str.contains("[{}-{}]".format(chr(lo), chr(hi)), regex=True, na=False)
            row[script] = int(mask.sum())
            accounted |= mask
        row["Other-NonASCII"] = int((non_ascii & ~accounted).sum())
        rows.append(row)
    return pd.DataFrame(rows).set_index("table")

Writing prep_final.py


In [3]:
sys.path.insert(0, str(DATA_DIR.resolve()))
import prep_final
importlib.reload(prep_final)
from prep_final import (add_source_column, normalize_country, process_business_name,
                        process_business_address, clean_business_name, _clean_address)
print(f"prep_final loaded — {prep_final.WORKERS} worker processes for the unique names / addresses")

prep_final loaded — 8 worker processes for the unique names / addresses


## 1. Load the raw data

In [4]:
t0 = time.time()
read = lambda p: pd.read_csv(p, sep="\t", dtype=str, keep_default_na=False, na_values=[""])
datasets = {
    "train_s1": read(TRAIN_DIR / "train_source1.tsv"),
    "train_s2": read(TRAIN_DIR / "train_source2.tsv"),
    "train_s3": read(TRAIN_DIR / "train_source3.tsv"),
    "ground_truth": read(TRAIN_DIR / "train_ground_truth.tsv"),
    "test_s1": read(TEST_DIR / "test_source1.tsv"),
    "test_s2": read(TEST_DIR / "test_source2.tsv"),
    "test_s3": read(TEST_DIR / "test_source3.tsv"),
}
for name, df in datasets.items():
    print(f"{name:13s}: {df.shape}")
print(f"[{time.time()-t0:.0f}s]")

train_s1     : (2206821, 4)
train_s2     : (5034616, 4)
train_s3     : (5285603, 4)
ground_truth : (2206821, 2)
test_s1      : (1732544, 4)
test_s2      : (4887273, 4)
test_s3      : (5082316, 4)
[22s]


## 2. Quick EDA

Nulls, duplicate ids, countries (France is test-only), and how many S2/S3 records one S1 entity has.

In [5]:
for name, df in datasets.items():
    nulls = df.isna().sum()
    dup = df["entity_id"].duplicated().sum() if "entity_id" in df else "-"
    print(f"{name:13s} rows {len(df):>10,}   duplicate ids {dup!s:>4}   nulls: "
          + ", ".join(f"{c} {v:,} ({v/len(df):.1%})" for c, v in nulls.items() if v))

print()
for name, df in datasets.items():
    if "country" in df:
        print(f"{name:13s}", df["country"].value_counts().to_dict())

gt = datasets["ground_truth"]
n_match = gt["matched_entity_ids"].fillna("").map(lambda x: len([i for i in x.split(",") if i.strip()]))
print("\nS2/S3 matches per S1 entity:", n_match.value_counts().sort_index().to_dict())
matched = set(i.strip() for x in gt["matched_entity_ids"].dropna() for i in x.split(","))
for name in ("train_s2", "train_s3"):
    print(f"{name}: {(~datasets[name]['entity_id'].isin(matched)).sum():,} records match no S1 (pure distractors)")

train_s1      rows  2,206,821   duplicate ids    0   nulls: 
train_s2      rows  5,034,616   duplicate ids    0   nulls: business_address 168,967 (3.4%)
train_s3      rows  5,285,603   duplicate ids    0   nulls: business_address 175,916 (3.3%)
ground_truth  rows  2,206,821   duplicate ids    -   nulls: matched_entity_ids 123,247 (5.6%)
test_s1       rows  1,732,544   duplicate ids    0   nulls: 
test_s2       rows  4,887,273   duplicate ids    0   nulls: business_address 129,408 (2.6%)
test_s3       rows  5,082,316   duplicate ids    0   nulls: business_address 136,098 (2.7%)

train_s1      {'US': 1323633, 'India': 883188}
train_s2      {'US': 3016817, 'India': 2017799}
train_s3      {'US': 3170056, 'India': 2115547}
test_s1       {'India': 809986, 'US': 663106, 'France': 259452}
test_s2       {'India': 2312565, 'US': 1871330, 'France': 703378}
test_s3       {'India': 2405000, 'US': 1945701, 'France': 731615}

S2/S3 matches per S1 entity: {0: 123247, 1: 119157, 2: 375212, 3: 530841, 4

## 3. The cleaning on real hard cases

Names and addresses taken from the blocking miss logs. Left = raw, right = what blocking now sees.

In [6]:
names = [
    "OUTSHlNE  FLAVOUR PRIVATE LIMITED", "Di6ital Estate", "-- Holloway Peak Inc Seafood",
    "Ellis Plumbing LLC", "Wallace Farms", "Cyril Motors", "CANDY'S-SERVICES", "Jacky’s Insurance",
    "odinternational.com", "www.trinity-construction.co.in", "Brixhalo t/a Creative Enterprises",
    "ABC Foods Pvt Ltd DBA Sharma Sweets Ltd", "रियल एग्रो प्राइवेट लिमिटेड",
    "सिटी हेल्थकेयर प्राइवेट लिमिटेड", "इंडो टेक्नोलॉजीज प्रा. लि.",
    "சில்வர் டெக்னாலஜீஸ் பிரைவேட் லிமிடெட்", "ওয়ান সফটওয়্যার প্রাইভেট লিমিটেড",
    "Vanguard Métropolitan Sunshine", "CafÃ© de Flore SARL", "L'Atelier d'Antoine SAS", "Dupont et Fils",
]
rows = []
for n in names:
    r = clean_business_name(n)
    rows.append({"raw": n, "name_core": r[1], "name_translit": r[0], "alias": " | ".join(x for x in r[15:] if x)})
display(pd.DataFrame(rows))

,raw,name_core,name_translit,alias
0,OUTSHlNE FLAVOUR PRIVATE LIMITED,outshine flavour,outshine flavour pvt ltd,
1,Di6ital Estate,digital estate,digital estate,
2,-- Holloway Peak Inc Seafood,holloway peak seafood,holloway peak inc seafood,
3,Ellis Plumbing LLC,ellis plumbing,ellis plumbing llc,
4,Wallace Farms,wallace farms,wallace farms,
5,Cyril Motors,cyril motors,cyril motors,
6,CANDY'S-SERVICES,candys services,candys services,
7,Jacky’s Insurance,jackys insurance,jackys insurance,
8,odinternational.com,odinternationalcom,odinternationalcom,
9,www.trinity-construction.co.in,trinityconstructioncom,trinityconstructioncom,


In [7]:
addrs = [
    "206 FLAT NO. LG 1 KH NO. 514, 516 SHYAM BHAWAN JHANDA COLONY FATEHPUR, DELHI, Delhi",
    "H.No 143 / B-4 Paschim Vihar, New Delhi, North Delhi, Delhi - 110063",
    "6309 EVANGELINE TRAIL, AUSTIN, TX 78701", "19034 Woodburn Road, Woodburn, IN",
    "001491 JOHN DAY DR, EUGENE, OR", "H.no 829 D-225vivek Vihar, Delhi, दिल्ली",
    "1161d Brookside Ave, Evansdale, Iowa", "H.NO ##6 , 1ST FLOOR, ANSUL PLAZA MARKET SHALIMAR BAGH, DELHI, दिल्ली",
    "Door No ##97 Lakshmi Vilasam, Pathanamthitta, കേരളം", "01160 Kirkland Avenue, Nashville, Tennessee",
    "4 Rue de la Constitution, Nantes, Pays de la Loire 44000", "N° 12 bis, Allée des Roses, 75011 Paris CEDEX",
    "null",
]
rows = []
for a in addrs:
    r = _clean_address(a)
    rows.append({"raw": a, "address_clean": r[0], "pin": r[1], "house": r[4], "unit": r[5], "street": r[11]})
display(pd.DataFrame(rows))

,raw,address_clean,pin,house,unit,street
0,"206 FLAT NO. LG 1 KH NO. 514, 516 SHYAM BHAWAN JHANDA COLONY FATEHPUR, DELHI, Delhi",206 flat no lg 1 kh no 514 516 shyam bldg jhanda colony fatehpur dl,,206,,
1,"H.No 143 / B-4 Paschim Vihar, New Delhi, North Delhi, Delhi - 110063",hno 143 b 4 paschim vihar dl n dl 110063,110063,143,,
2,"6309 EVANGELINE TRAIL, AUSTIN, TX 78701",6309 evangeline trl austin tx 78701,78701,6309,,evangeline trl
3,"19034 Woodburn Road, Woodburn, IN",19034 woodburn rd woodburn in,,19034,,woodburn rd
4,"001491 JOHN DAY DR, EUGENE, OR",1491 john day dr eugene or,,1491,,john day dr
5,"H.no 829 D-225vivek Vihar, Delhi, दिल्ली",hno 829 d 225 vivek vihar dl,,829,,
6,"1161d Brookside Ave, Evansdale, Iowa",1161 d brookside ave evansdale ia,,1161,,brookside ave
7,"H.NO ##6 , 1ST FLOOR, ANSUL PLAZA MARKET SHALIMAR BAGH, DELHI, दिल्ली",hno 6 1 fl ansul plaza market shalimar bagh dl xx6,,6,,
8,"Door No ##97 Lakshmi Vilasam, Pathanamthitta, കേരളം",door no 97 lakshmi vilasam pathanamthitta kl xx97,,,97,
9,"01160 Kirkland Avenue, Nashville, Tennessee",1160 kirkland ave nashville tn,,1160,,kirkland ave


## 4. Clean every table

Names and addresses are cleaned once per **unique** string, in parallel worker processes
(`prep_final.WORKERS`). Expect roughly 10-25 minutes for all seven tables.

In [8]:
for name in ["train_s1", "train_s2", "train_s3", "test_s1", "test_s2", "test_s3"]:
    t = time.time()
    df = datasets[name]
    df = add_source_column(df)
    df = normalize_country(df)
    df = process_business_name(df)
    df = process_business_address(df)
    datasets[name] = df
    gc.collect()
    print(f"{name:9s}: {len(df):>10,} rows, {df.shape[1]} columns   [{time.time()-t:.0f}s]", flush=True)

train_s1 :  2,206,821 rows, 50 columns   [19s]
train_s2 :  5,034,616 rows, 50 columns   [51s]
train_s3 :  5,285,603 rows, 50 columns   [55s]
test_s1  :  1,732,544 rows, 50 columns   [17s]
test_s2  :  4,887,273 rows, 50 columns   [59s]
test_s3  :  5,082,316 rows, 50 columns   [77s]


## 5. Verification

In [9]:
cols = ["business_name", "name_core", "business_address", "address_clean", "address_pin", "house_number"]
display(datasets["train_s2"][cols].sample(12, random_state=1))

print(f"\n{'table':10s} {'pin found':>10s} {'no address':>11s} {'web name':>9s} {'alias':>7s} {'empty name':>11s}")
for name in ["train_s1", "train_s2", "train_s3", "test_s1", "test_s2", "test_s3"]:
    df = datasets[name]
    print(f"{name:10s} {(df['address_pin'] != '').mean():10.1%} {df['is_missing_address'].mean():11.1%} "
          f"{df['name_core'].str.endswith('com').mean():9.2%} {df['has_alias'].mean():7.2%} "
          f"{(df['name_core'] == '').mean():11.2%}")

# every ground-truth match must be same-country (country is a hard blocking filter)
pairs = (gt.dropna(subset=["matched_entity_ids"])
           .assign(cand=lambda d: d["matched_entity_ids"].str.split(","))
           .explode("cand")[["source1_entity_id", "cand"]])
pairs["cand"] = pairs["cand"].str.strip()
c1 = datasets["train_s1"].set_index("entity_id")["country_clean"]
c23 = pd.concat([datasets["train_s2"], datasets["train_s3"]]).set_index("entity_id")["country_clean"]
same = pairs["source1_entity_id"].map(c1).to_numpy() == pairs["cand"].map(c23).to_numpy()
print(f"\ntrue pairs: {len(pairs):,}   cross-country: {(~same).sum():,}   candidate id missing: "
      f"{pairs['cand'].map(c23).isna().sum():,}")

,business_name,name_core,business_address,address_clean,address_pin,house_number
293730,Royal Safe Paceline Inc,royal safe paceline,"HOMELAND AVE, NULL, NORMAN, OK",homeland ave norman ok,,
4038314,Heritage Éducation Products,heritage education products,"7582 EAST POINT DOUGLAS RD, COTTAGE GROVE, MN",7582 e pt douglas rd cottage grove mn,,7582
1071002,Knight Educational Society Private Limited,knight educational society,"Karnataka, 50 , KNO 499/499, BCMC HBCS, RAGHUVANAHALLI, BANGALORE SOUTH",ka 50 kno 499 bcmc hbcs raghuvanahalli bangalore s,,
691119,HART VALLEY LÍBERTY CENTER,hart valley liberty center,"FOURTEENTH STREET, SAN BERNARDINO TOOWNSHIP, CA",fourteenth st san bernardino toownship ca,,
2554869,Rainbow Partners Center,rainbow partners center,"Gujarat, 401-402, SHREEYA ATLANTA, NR SAHJANAND BUNGLOW, AHMADABAD CITY",gj 401 402 shreeya atlanta nr sahjanand bunglow ahmadabad city,,
4604271,ईस्ट डेवलपर्स प्राइवेट लिमिटेड,ista devalaparsa,"BHAGAT HOUSE, JOGESHWARI EAST, MUMBAI, Maharashtra",bhagat hno jogeshwari e mumbai mh,,
194709,"PC Green, Critsba, O.D., P.C.",pc green critsba od pc,"86 Searcy Lane, STALEY, NC",86 searcy ln staley nc,,86
1969951,tharpglobal.com,tharpglobalcom,"MANCHESTER, TN, 437 WOODROW ST",manchester tn 437 woodrow st,,
2157663,ब्राइट मैनेजमेंट लिमिटेड,braita mainejamemta,"BARANA 6805 PALERA, TIKAMGARH, Madhya Pradesh",barana 6805 palera tikamgarh mp,,
4517414,First Highland Siddhi LP,first highland siddhi,"OH, 465 BROAD MEADOWS BLVD, COLUMBUS",oh 465 broad meadows blvd columbus,,



table       pin found  no address  web name   alias  empty name
train_s1         1.1%        0.0%     0.08%   0.00%       0.00%
train_s2         1.4%        3.4%     3.92%   0.00%       0.00%
train_s3         1.4%        3.3%     3.91%   2.49%       0.00%
test_s1          0.8%        0.0%     0.08%   0.00%       0.00%
test_s2          1.1%        2.6%     3.14%   0.00%       0.00%
test_s3          1.1%        2.7%     3.18%   1.85%       0.00%

true pairs: 7,638,365   cross-country: 0   candidate id missing: 0


## 6. How findable are the true pairs after cleaning?

For every true (S1, candidate) pair: does an **exact** key already agree? (same `name_core`, same name
without spaces, same order-free phonetic skeleton, DBA / alias half, same house number, same postal code).
Pairs with no exact name key and no shared address word are the ones blocking must find by fuzzy keys.
This runs on the ground truth only (no model), and is the quickest check that a cleaning change helped.

In [10]:
t = time.time()
kc = ["entity_id", "name_core", "name_compact", "name_skeleton", "name_primary", "name_alias",
      "house_number", "address_pin", "address_tokens_sorted", "is_missing_address"]
S = datasets["train_s1"][kc].rename(columns=lambda c: c + "_s")
C = pd.concat([datasets["train_s2"][kc], datasets["train_s3"][kc]], ignore_index=True).rename(columns=lambda c: c + "_c")
M = (pairs.merge(S, left_on="source1_entity_id", right_on="entity_id_s")
          .merge(C, left_on="cand", right_on="entity_id_c"))
eq = lambda a, b: (M[a] == M[b]) & (M[a] != "")
keys = {
    "same name_core": eq("name_core_s", "name_core_c"),
    "same name without spaces": eq("name_compact_s", "name_compact_c"),
    "same phonetic skeleton (any order)": eq("name_skeleton_s", "name_skeleton_c"),
    "alias half = other name": (((M["name_alias_s"] != "") & ((M["name_alias_s"] == M["name_core_c"]) | (M["name_primary_s"] == M["name_core_c"])))
                                | ((M["name_alias_c"] != "") & ((M["name_alias_c"] == M["name_core_s"]) | (M["name_primary_c"] == M["name_core_s"])))),
    "same house number": eq("house_number_s", "house_number_c"),
    "same postal code": eq("address_pin_s", "address_pin_c"),
}
name_any = keys["same name_core"] | keys["same name without spaces"] | keys["same phonetic skeleton (any order)"] | keys["alias half = other name"]
keys["ANY exact name key"] = name_any
rep = pd.DataFrame({"share of true pairs": {k: v.mean() for k, v in keys.items()}})
rep["pairs"] = {k: int(v.sum()) for k, v in keys.items()}
display(rep.style.format({"share of true pairs": "{:.4f}", "pairs": "{:,}"}))

# pairs with no exact name key: do they at least share an address word?
hard = M.loc[~name_any.to_numpy()]
smp = hard.sample(min(len(hard), 300_000), random_state=0)
shared = [len(set(a.split()) & set(b.split())) for a, b in zip(smp["address_tokens_sorted_s"], smp["address_tokens_sorted_c"])]
shared = np.array(shared)
print(f"no exact name key: {len(hard):,} pairs ({len(hard)/len(M):.2%}).  Of those (sample of {len(smp):,}):")
print(f"   candidate has no address ........ {smp['is_missing_address_c'].mean():.1%}")
print(f"   share >= 2 address words ........ {(shared >= 2).mean():.1%}")
print(f"   share no address word at all .... {(shared == 0).mean():.1%}")
display(smp.loc[shared == 0, ["name_core_s", "name_core_c", "address_tokens_sorted_s", "address_tokens_sorted_c"]].head(15))
del S, C, M, hard, smp, keys, name_any
gc.collect()
print(f"[{time.time()-t:.0f}s]")

,share of true pairs,pairs
same name_core,0.5253,"4,012,262"
same name without spaces,0.5288,"4,039,073"
same phonetic skeleton (any order),0.5829,"4,452,165"
alias half = other name,0.0172,"131,754"
same house number,0.4528,"3,458,558"
same postal code,0.0026,"19,514"
ANY exact name key,0.6018,"4,597,018"


no exact name key: 3,041,347 pairs (39.82%).  Of those (sample of 300,000):
   candidate has no address ........ 3.9%
   share >= 2 address words ........ 96.1%
   share no address word at all .... 3.9%


,name_core_s,name_core_c,address_tokens_sorted_s,address_tokens_sorted_c
1517046,pediatric dentistry medicine of louisville,pediatric of mecdine dentistry louisville,304 ky louisville n shawnee ter,
3565519,ds and sons,ds and sons phroave,6 bangalore hosur ka mile roadbommanahalli,
2763764,lucknow rubber,lucknow rlafr,570 867 alambagh gopalpuri lucknow up,
3789363,crystal realty group,crystal realty,5411 ghost given hollow rd wv,
1914649,basalt armada,basalt acrmcmaa,21 6006 bristol ny s,
2333250,ridgeline,ridgeline ventures,30 blueberry dunn hill ln nc,
4384689,downtown bakery center,downtown bakery center 8789434468,360 beaumont china st tx,
2310168,cardiology custom physicians,cardiology custom physicians commission,1486 655 fm magnolia tx,
5294956,united society,united society enterprises,609 kings mountain mountain nc st,
1300225,piedmont committee,piedmont committee federation,7805 clinton ln md rhonda,


[21s]


## 7. Train / validation split

At the S1-entity level, stratified by whatever countries the data contains, so all pairs of one entity
stay on one side. Blocking ignores it; the matching model uses it.

In [11]:
RANDOM_STATE, VAL_FRACTION = 42, 0.15
s1_ids = datasets["train_s1"][["entity_id", "country_clean"]].reset_index(drop=True)
rng = np.random.default_rng(RANDOM_STATE)
val_mask = np.zeros(len(s1_ids), dtype=bool)
for _, group in s1_ids.groupby("country_clean"):
    if len(group) < 4:
        continue
    val_mask[rng.choice(group.index.to_numpy(), size=max(1, round(len(group) * VAL_FRACTION)), replace=False)] = True
s1_ids["split"] = np.where(val_mask, "val", "train")
print(s1_ids.groupby(["country_clean", "split"]).size().unstack(fill_value=0))

split            train     val
country_clean                 
India           750710  132478
US             1125088  198545


## 8. Save the artifacts

In [12]:
t = time.time()
for name, df in datasets.items():
    path = ART / f"{name}_clean.csv"
    df.to_csv(path, index=False)
    print(f"  {name:13s} -> {path}  ({os.path.getsize(path)/2**20:,.0f} MB, {len(df):,} rows, {df.shape[1]} cols)", flush=True)
s1_ids[["entity_id", "split"]].to_csv(ART / "train_val_split.csv", index=False)
print(f"  {'split':13s} -> {ART / 'train_val_split.csv'}")
print(f"[{time.time()-t:.0f}s]")

print(f"\n{'table':13s} {'rows':>10s} {'cols':>5s}")
for name, df in datasets.items():
    print(f"{name:13s} {len(df):10,d} {df.shape[1]:5d}")
print("\ncolumns:", datasets["train_s1"].columns.tolist())

  train_s1      -> artifacts/train_s1_clean.csv  (965 MB, 2,206,821 rows, 50 cols)
  train_s2      -> artifacts/train_s2_clean.csv  (2,183 MB, 5,034,616 rows, 50 cols)
  train_s3      -> artifacts/train_s3_clean.csv  (2,268 MB, 5,285,603 rows, 50 cols)
  ground_truth  -> artifacts/ground_truth_clean.csv  (125 MB, 2,206,821 rows, 2 cols)
  test_s1       -> artifacts/test_s1_clean.csv  (772 MB, 1,732,544 rows, 50 cols)
  test_s2       -> artifacts/test_s2_clean.csv  (2,191 MB, 4,887,273 rows, 50 cols)
  test_s3       -> artifacts/test_s3_clean.csv  (2,230 MB, 5,082,316 rows, 50 cols)
  split         -> artifacts/train_val_split.csv
[195s]

table               rows  cols
train_s1       2,206,821    50
train_s2       5,034,616    50
train_s3       5,285,603    50
ground_truth   2,206,821     2
test_s1        1,732,544    50
test_s2        4,887,273    50
test_s3        5,082,316    50

columns: ['entity_id', 'business_name', 'business_address', 'country', 'source', 'country_clean', 'name_t

---
**Next:** `python blocking_final.py` (train), then `python blocking_final.py test`.

In `final.py` the test files are already cleaned by this notebook (stage 1 is skipped because they
are newer than the raw files). To keep a forced re-run identical, change its stage-1 import from
`from utils import (...)` to `from prep_final import (...)`, and `import blocking` to
`import blocking_final as blocking`.